# Predicting the hate speech-score of Bluesky posts using machine learning
### 4982 - Data Processing 2
##### By Team 7 - by  Maximilian Dieringer, Matthäus Engelbrecht,  Borisz Gabor Sztaniszlav, Martin Sepulveda

### Project Overview
This project aims to predict the hate speech score of social media posts using machine learning. We use a labeled dataset from Huggingface to train our models and test them on real posts from the Bluesky platform. To achieve  this, we build a scalable data processing pipeline with Apache Spark. The data is cleaned, prepared, and transformed so it can be used for training. After that, we evaluate different models and use the best one to predict how much hate speech is in new, unseen content.

**Steps:**   
To reach our goal, we followed a structured approach using Apache Spark for scalable data processing and model training.The following steps were carried out and are explained in detail in the next sections:

1. Spark Setup
2. Load Huggingface Dataset
3. Preprocessing Huggingface Dataset
4. Prepare NLP Resources  
5. Register Preprocessing Functions in Preproc
6. NLP for Huggingface Dataset  
7. Loading Bluesky API Data
8. Preprocessing Bluesky API Data
9. NLP for Bluesky API Data
10. Preprocessing for Machine Learning
11. Machine Learning
12. Model Evaluation
13. Bluesky Predictions

### 1) Spark Setup
We begin by setting up Apache Spark. Spark allows us to handle data in a scalable way, which is important when working with big datasets and machine learning models. To run Spark inside a Jupyter Notebook, we first set the correct path to where Spark is installed on the system. This depends on the operating system and needs to be adapted if the code is run on another machine. Next, we create a SparkSession. This object manages how Spark runs and allows us to use DataFrames, which are similar to tables in databases. We also initialize a SparkContext, which gives us access to Spark features that we will need later in the project. Finally, we create a SQLContext to work with structured data more easily.

The code used here was provided in the course *Data Processing 2: Scalable Data Processing, Legal & Ethical Foundations of Data Science* (WU Vienna), by Ines Akaichi, and has been adapted for our project.

**Libraries used in this step:**
- `findspark` – to link Spark with Jupyter
- `pyspark.sql.SparkSession` – to manage the Spark process
- `pyspark.sql.SQLContext` – to work with structured data

In [2]:
# Load the findspark module to connect local Python with Spark
import findspark

# Try to initialize findspark
try:
    findspark.init("/usr/local/spark/")
    print("findspark initialized")
except Exception as e:
    print(f"An error occurred during findspark initialization: {e}")


# Import the SparkSession and SQLContext modules
from pyspark.sql import SparkSession
from pyspark.sql import SQLContext

# Build the SparkSession to run locally with 1GB executor memory
spark = SparkSession.builder \
   .master("local") \
   .appName("NLP classification tweets") \
   .config("spark.executor.memory", "1gb") \
   .getOrCreate()

# Create SparkContext to manage the connection to the Spark cluster and access lower-level Spark functions
sc = spark.sparkContext

# Initialize SQLContext to enable working with structured data using SQL-like operations
sqlContext = SQLContext(sc)

findspark initialized


### 2) Load Huggingface Dataset
In this step, we load the labeled dataset that we use to train our machine learning models. The data is retrieved directly from Huggingface using the datasets library and contains thousands of social media comments labeled with a continuous hate speech score.

The code used to load the data is provided on the official Huggingface dataset page: https://huggingface.co/datasets/ucberkeley-dlab/measuring-hate-speech. After loading, we save the data locally as a CSV file to make it easier to load into Spark. Finally, we read the CSV into a Spark DataFrame.

**Libraries used:**
- `datasets` – to access the labeled dataset from Huggingface
- `pyspark.sql` – to load the CSV into a Spark DataFrame

In [1]:
# Install Huggingface datasets library to load datasets directly
!pip install datasets #Just need to execute this code one time

# Import the HuggingFace datasets library
from datasets import load_dataset

# Try to load the 'train' split from the dataset
try:
    ds = load_dataset("ucberkeley-dlab/measuring-hate-speech", split="train")
    print("Huggingface dataset loaded successfully")
except Exception as e:
    print(f"An error occurred while loading the dataset: {e}")

# Save the dataset as CSV
ds.to_csv("hate_speech_data.csv", index=False)

# Try to load the CSV with Spark
try:
    df = spark.read.csv("hate_speech_data.csv", header=True, inferSchema=True)
    print("CSV file loaded into Spark DataFrame successfully")
except Exception as e:
    print(f"An error occurred while reading the CSV file with Spark: {e}")

     |████████████████████████████████| 480 kB 9.5 MB/s eta 0:00:01
     |████████████████████████████████| 78 kB 69.1 MB/s eta 0:00:01
     |████████████████████████████████| 64 kB 66.2 MB/s eta 0:00:01
     |████████████████████████████████| 132 kB 88.8 MB/s eta 0:00:01
     |████████████████████████████████| 40.0 MB 61.9 MB/s eta 0:00:01██                         | 8.9 MB 61.9 MB/s eta 0:00:01
     |████████████████████████████████| 194 kB 106.1 MB/s eta 0:00:01
     |████████████████████████████████| 179 kB 51.5 MB/s eta 0:00:01
     |████████████████████████████████| 1.3 MB 54.9 MB/s eta 0:00:01
     |████████████████████████████████| 515 kB 81.4 MB/s eta 0:00:01
     |████████████████████████████████| 147 kB 84.0 MB/s eta 0:00:01
     |████████████████████████████████| 243 kB 56.6 MB/s eta 0:00:01
     |████████████████████████████████| 129 kB 69.6 MB/s eta 0:00:01
     |████████████████████████████████| 319 kB 74.8 MB/s eta 0:00:01
     |████████████████████████████████| 3.1 MB 

Creating CSV from Arrow format:   0%|          | 0/136 [00:00<?, ?ba/s]

An error occurred while reading the CSV file with Spark: name 'spark' is not defined


### 3) Preprocessing Huggingface Dataset
To make the preprocessing more efficient, we first reduce the dataset size by working with a random subset. This helps speed up development and keeps the runtime manageable. We then only select comment_id, text, and hate_speech_score because these are the only columns relevant for our NLP and prediction task. All other columns are excluded to reduce complexity and protect sensitive or personal information. Finally, we remove any rows that contain missing values to ensure the data is clean and consistent for the next processing steps.

In [3]:
# Display schema and first row to check structure and data format
df.printSchema()
df.show(1)

25/07/06 19:56:14 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


root
 |-- comment_id: integer (nullable = true)
 |-- annotator_id: integer (nullable = true)
 |-- platform: integer (nullable = true)
 |-- sentiment: double (nullable = true)
 |-- respect: double (nullable = true)
 |-- insult: double (nullable = true)
 |-- humiliate: double (nullable = true)
 |-- status: double (nullable = true)
 |-- dehumanize: double (nullable = true)
 |-- violence: double (nullable = true)
 |-- genocide: double (nullable = true)
 |-- attack_defend: double (nullable = true)
 |-- hatespeech: double (nullable = true)
 |-- hate_speech_score: double (nullable = true)
 |-- text: string (nullable = true)
 |-- infitms: string (nullable = true)
 |-- outfitms: string (nullable = true)
 |-- annotator_severity: string (nullable = true)
 |-- std_err: string (nullable = true)
 |-- annotator_infitms: string (nullable = true)
 |-- annotator_outfitms: string (nullable = true)
 |-- hypothesis: string (nullable = true)
 |-- target_race_asian: string (nullable = true)
 |-- target_race_

In [4]:
# Split dataset into smaller part, so pre-processing can be done faster
splits = df.randomSplit([0.5, 0.5], seed = 123)
df_not_work = splits[0]
df_work = splits[1]

df = df_work

In [5]:
# Try to select all the relevant columns from the dataset
try:
    selected_df = df.select("comment_id", "text", "hate_speech_score")
    print("Columns selected successfully.")
except Exception as e:
    print(f"Error selecting columns from DataFrame: {e}")

Columns selected successfully.


In [6]:
selected_df.printSchema()
selected_df.show(1)

root
 |-- comment_id: integer (nullable = true)
 |-- text: string (nullable = true)
 |-- hate_speech_score: double (nullable = true)



+----------+--------------------+-----------------+
|comment_id|                text|hate_speech_score|
+----------+--------------------+-----------------+
|         1|White people are ...|             0.46|
+----------+--------------------+-----------------+
only showing top 1 row


In [7]:
# Remove all rows with null values in any of the selected columns
df_cleaned = selected_df.dropna()

# Count rows before and after removing nulls
original_count = selected_df.count()
cleaned_count = df_cleaned.count()

print(f"Rows before removing nulls: {original_count}")
print(f"Rows after removing nulls: {cleaned_count}")

Rows before removing nulls: 67768
Rows after removing nulls: 67768


In [8]:
data_df = df_cleaned

### 4) Prepare NLP Resources
In this step, we prepare the language tools needed for later text processing. These include components for language detection, part-of-speech tagging, stopword removal, and lemmatization. Installing and loading these tools ensures that the following NLP steps can run smoothly without interruption.

The code used in this part is based on the material provided in the course *Data Processing 2: Scalable Data Processing, Legal & Ethical Foundations of Data Science* (WU Vienna) by Ines Akaichi.

**Libraries used:**  
- `nltk` – main library for natural language processing
- `langid` – for detecting the language of comments

In [2]:
# Install a pip package in the current Jupyter kernel
import sys
!{sys.executable} -m pip install nltk --no-cache-dir

     |████████████████████████████████| 1.5 MB 10.2 MB/s eta 0:00:01
     |████████████████████████████████| 785 kB 95.3 MB/s eta 0:00:01


In [3]:
# Install a pip package in the current Jupyter kernel
import sys
!{sys.executable} -m pip install langid --no-cache-dir

     |████████████████████████████████| 1.9 MB 20.8 MB/s eta 0:00:01
  Created wheel for langid: filename=langid-1.1.6-py3-none-any.whl size=1941189 sha256=c7ef3ac73d29544d07405d776cdc1f8facdc924b17a6cabb5dfcf5f23b1909cf
  Stored in directory: /tmp/pip-ephem-wheel-cache-nsr24u17/wheels/c5/01/a4/0160c55074707b535a6757a541842817d530d8080ca943a107
Successfully built langid


In [11]:
# Download averaged_perceptron_tagger from the nltk
import nltk
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/matthausengelbrecht/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [12]:
# Download en averaged_perceptron_tagger from the nltk
import nltk
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/matthausengelbrecht/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [13]:
# Download stopwords from the nltk
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/matthausengelbrecht/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [14]:
# Download wordnet from the nltk
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/matthausengelbrecht/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

### 5) Register Preprocessing Functions in Preproc
In this step, we register all user-defined preprocessing functions so they can be used within Spark. These functions handle key tasks such as language detection, stopword removal, feature cleaning, POS-tagging, lemmatization, and checking for empty or blank text entries. Registering them in advance allows us to apply them later in a distributed way across the dataset.

All functions are included in the file `preproc.py`, which was provided to us as part of the course *Data Processing 2: Scalable Data Processing, Legal & Ethical Foundations of Data Science* (WU Vienna) by Ines Akaichi. We used this file without modification and integrated it into our pipeline to ensure consistency with the taught preprocessing structure.

**Libraries used:**  
- `pyspark.sql.functions.udf` – for registering Spark UDFs  
- `pyspark.sql.types.StringType` – to define return types  
- custom module `preproc` – contains all text processing logic

In [15]:
# Import modules needed for Spark UDFs and preprocessing functions
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Try to import custom functions and define Spark UDFs
try:
    import preproc as pp

    # Register all preprocessing functions as Spark UDFs
    # Each UDF takes a string input and applies a specific text processing step
    
    # Detects the language of a given input string
    check_lang_udf = udf(pp.check_lang, StringType())

    # Removes common stopwords from the input text
    remove_stops_udf = udf(pp.remove_stops, StringType())

    # Filters out features like links or irrelevant words
    remove_features_udf = udf(pp.remove_features, StringType())

    # Applies POS tagging and removes unwanted word categories
    tag_and_remove_udf = udf(pp.tag_and_remove, StringType())

    # Converts words to their lemmatized (base) form
    lemmatize_udf = udf(pp.lemmatize, StringType())

    # Checks if a text string is blank or contains only whitespace
    check_blanks_udf = udf(pp.check_blanks, StringType())

except Exception as e:
    print(f"An error occurred while setting up preprocessing UDFs: {e}")

### 6) NLP for Huggingface Dataset
In this step, we apply all registered functions to our dataset. This includes language filtering, removing stopwords and special characters, part-of-speech tagging, lemmatization, and handling empty or duplicate entries. At the end, we keep only the cleaned text and the corresponding hate speech score to prepare the data for machine learning.

The structure and logic of this step were provided in the course *Data Processing 2: Scalable Data Processing, Legal & Ethical Foundations of Data Science* (WU Vienna) by Ines Akaichi and were adapted to fit the needs of our project setup.

In [16]:
# Use "check_lang_udf" to predict the language of each comment
# The result is stored in a new column called "lang" using withColumn
# This allows us to keep the original text while adding the language prediction for further filtering
lang_df = data_df.withColumn("lang", check_lang_udf(data_df["text"]))

# Filter the dataset to keep only entries classified as English
# Only rows with at least 90 percent probability for English are retained
en_df = lang_df.filter(lang_df["lang"] == "en")

In [17]:
# Display 10 rows from the dataset with predicted language
lang_df.show(n=10)

# Count how many comments were classified as English
print("Number of English comments:", en_df.count())

# Show 2 example rows from the filtered English-only dataset
en_df.show(n=2)

+----------+--------------------+-----------------+----+
|comment_id|                text|hate_speech_score|lang|
+----------+--------------------+-----------------+----+
|         1|White people are ...|             0.46|  en|
|         1|White people are ...|             0.46|  en|
|         2|You've caught the...|             0.03|  en|
|         3|"I'm not saying y...|            -1.29|  en|
|         3|"I'm not saying y...|            -1.29|  en|
|         3|"I'm not saying y...|            -1.29|  en|
|         5|Fabrice Fabrice i...|            -2.84|  en|
|         5|Fabrice Fabrice i...|            -2.84|  en|
|         5|Fabrice Fabrice i...|            -2.84|  en|
|         6|First off you loo...|             1.72|  en|
+----------+--------------------+-----------------+----+
only showing top 10 rows


Number of English comments: 63230


+----------+--------------------+-----------------+----+
|comment_id|                text|hate_speech_score|lang|
+----------+--------------------+-----------------+----+
|         1|White people are ...|             0.46|  en|
|         1|White people are ...|             0.46|  en|
+----------+--------------------+-----------------+----+
only showing top 2 rows


In [18]:
# Use 'remove_stops_udf' to create a new column without stopwords
# This reduces dimensionality by removing common words that carry little meaning
# We use withColumn to add the new cleaned text column to the DataFrame
rm_stops_df = en_df.withColumn("stop_text", remove_stops_udf(en_df["text"]))

In [19]:
# Display original and cleaned text to verify stopword removal
rm_stops_df.select("text","stop_text").show(n=2, truncate=0)

Traceback (most recent call last):                                  (0 + 1) / 1]
  File "/opt/homebrew/Cellar/apache-spark/4.0.0/libexec/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
    code = worker(sock, authenticated)
  File "/opt/homebrew/Cellar/apache-spark/4.0.0/libexec/python/lib/pyspark.zip/pyspark/daemon.py", line 81, in worker
    worker_main(infile, outfile)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/apache-spark/4.0.0/libexec/python/lib/pyspark.zip/pyspark/worker.py", line 2068, in main
    if read_int(infile) == SpecialLengths.END_OF_STREAM:
       ~~~~~~~~^^^^^^^^
  File "/opt/homebrew/Cellar/apache-spark/4.0.0/libexec/python/lib/pyspark.zip/pyspark/serializers.py", line 597, in read_int
    raise EOFError
EOFError


+-----------------------+-------------------+
|text                   |stop_text          |
+-----------------------+-------------------+
|White people are trash.|White people trash.|
|White people are trash.|White people trash.|
+-----------------------+-------------------+
only showing top 2 rows


In [20]:
# Use 'remove_features_udf' to clean up additional unwanted tokens 
# This includes for example custom patterns like URLs or user mentions to reduce noise
# We use withColumn to add the new cleaned text column to the DataFrame
rm_features_df = rm_stops_df.withColumn("feat_text", remove_features_udf(rm_stops_df["stop_text"]))

In [21]:
# Display text before and after feature removal to verify cleaning step
rm_features_df.select("stop_text","feat_text").show(n=2, truncate=0)

Traceback (most recent call last):                                  (0 + 1) / 1]
  File "/opt/homebrew/Cellar/apache-spark/4.0.0/libexec/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
    code = worker(sock, authenticated)
  File "/opt/homebrew/Cellar/apache-spark/4.0.0/libexec/python/lib/pyspark.zip/pyspark/daemon.py", line 81, in worker
    worker_main(infile, outfile)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/apache-spark/4.0.0/libexec/python/lib/pyspark.zip/pyspark/worker.py", line 2068, in main
    if read_int(infile) == SpecialLengths.END_OF_STREAM:
       ~~~~~~~~^^^^^^^^
  File "/opt/homebrew/Cellar/apache-spark/4.0.0/libexec/python/lib/pyspark.zip/pyspark/serializers.py", line 597, in read_int
    raise EOFError
EOFError


+-------------------+------------------+
|stop_text          |feat_text         |
+-------------------+------------------+
|White people trash.|white people trash|
|White people trash.|white people trash|
+-------------------+------------------+
only showing top 2 rows


In [22]:
# Use 'tag_and_remove_udf' to tag remaining words and keep only nouns, verbs and adjectives
# We use withColumn to add the POS-tagged and filtered text as a new column
tagged_df = rm_features_df.withColumn("tagged_text", tag_and_remove_udf(rm_features_df["feat_text"]))

In [23]:
# Display text before and after POS tagging to verify filtering of unwanted word types
tagged_df.select("feat_text","tagged_text").show(n=2, truncate=0)

Traceback (most recent call last):                                  (0 + 1) / 1]
  File "/opt/homebrew/Cellar/apache-spark/4.0.0/libexec/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
    code = worker(sock, authenticated)
  File "/opt/homebrew/Cellar/apache-spark/4.0.0/libexec/python/lib/pyspark.zip/pyspark/daemon.py", line 81, in worker
    worker_main(infile, outfile)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/apache-spark/4.0.0/libexec/python/lib/pyspark.zip/pyspark/worker.py", line 2068, in main
    if read_int(infile) == SpecialLengths.END_OF_STREAM:
       ~~~~~~~~^^^^^^^^
  File "/opt/homebrew/Cellar/apache-spark/4.0.0/libexec/python/lib/pyspark.zip/pyspark/serializers.py", line 597, in read_int
    raise EOFError
EOFError


+------------------+--------------------+
|feat_text         |tagged_text         |
+------------------+--------------------+
|white people trash| white people trash |
|white people trash| white people trash |
+------------------+--------------------+
only showing top 2 rows


In [24]:
# Use 'lemmatize_udf' to reduce words to their base forms for better consistency
# This helps reduce dimensionality and improves feature alignment across similar terms
# We use withColumn to add the lemmatized version of the tagged text to the DataFrame
lemm_df = tagged_df.withColumn("lemm_text", lemmatize_udf(tagged_df["tagged_text"]))

In [25]:
# Display text before and after lemmatization to verify word normalization
lemm_df.select("tagged_text","lemm_text").show(n=2, truncate=0)

Traceback (most recent call last):                                  (0 + 1) / 1]
  File "/opt/homebrew/Cellar/apache-spark/4.0.0/libexec/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
    code = worker(sock, authenticated)
  File "/opt/homebrew/Cellar/apache-spark/4.0.0/libexec/python/lib/pyspark.zip/pyspark/daemon.py", line 81, in worker
    worker_main(infile, outfile)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/apache-spark/4.0.0/libexec/python/lib/pyspark.zip/pyspark/worker.py", line 2068, in main
    if read_int(infile) == SpecialLengths.END_OF_STREAM:
       ~~~~~~~~^^^^^^^^
  File "/opt/homebrew/Cellar/apache-spark/4.0.0/libexec/python/lib/pyspark.zip/pyspark/serializers.py", line 597, in read_int
    raise EOFError
EOFError


+--------------------+------------------+
|tagged_text         |lemm_text         |
+--------------------+------------------+
| white people trash |white people trash|
| white people trash |white people trash|
+--------------------+------------------+
only showing top 2 rows


Traceback (most recent call last):
  File "/opt/homebrew/Cellar/apache-spark/4.0.0/libexec/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
    code = worker(sock, authenticated)
  File "/opt/homebrew/Cellar/apache-spark/4.0.0/libexec/python/lib/pyspark.zip/pyspark/daemon.py", line 81, in worker
    worker_main(infile, outfile)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/apache-spark/4.0.0/libexec/python/lib/pyspark.zip/pyspark/worker.py", line 2068, in main
    if read_int(infile) == SpecialLengths.END_OF_STREAM:
       ~~~~~~~~^^^^^^^^
  File "/opt/homebrew/Cellar/apache-spark/4.0.0/libexec/python/lib/pyspark.zip/pyspark/serializers.py", line 597, in read_int
    raise EOFError
EOFError


In [26]:
# Use 'check_blanks_udf' to detect rows with only blank spaces in the lemmatized text
# This step helps ensure that no empty or meaningless inputs are passed into later ML processing
# We use withColumn to add a helper column and then filter out the blank rows accordingly
check_blanks_df = lemm_df.withColumn("is_blank", check_blanks_udf(lemm_df["lemm_text"]))
no_blanks_df = check_blanks_df.filter(check_blanks_df["is_blank"] == "False")

In [27]:
# Display 'is_blank' values to verify all empty rows were removed
no_blanks_df.select("is_blank").show(n=2, truncate=0)

+--------+
|is_blank|
+--------+
|False   |
|False   |
+--------+
only showing top 2 rows


In [28]:
# Use 'dropDuplicates' to remove duplicate rows from the DataFrame  
# This is based on identical 'text' and 'hate_speech_score' values  
# Ensures data quality by avoiding multiple entries of the same sample
dedup_df = no_blanks_df.dropDuplicates(['text', 'hate_speech_score'])

In [29]:
# Display final schema to verify structure and included columns after cleaning steps
dedup_df.printSchema()

root
 |-- comment_id: integer (nullable = true)
 |-- text: string (nullable = true)
 |-- hate_speech_score: double (nullable = true)
 |-- lang: string (nullable = true)
 |-- stop_text: string (nullable = true)
 |-- feat_text: string (nullable = true)
 |-- tagged_text: string (nullable = true)
 |-- lemm_text: string (nullable = true)
 |-- is_blank: string (nullable = true)



In [30]:
# Use 'select' to create a final dataset with only the relevant columns 
# This includes comment_id as identifier, lemmatized text for analysis, and the hate speech score as target
data_set = dedup_df.select(dedup_df['comment_id'], dedup_df['lemm_text'], dedup_df['hate_speech_score'])

In [31]:
# Display the schema of the final dataset to verify column names and data types
data_set.printSchema()

root
 |-- comment_id: integer (nullable = true)
 |-- lemm_text: string (nullable = true)
 |-- hate_speech_score: double (nullable = true)



In [32]:
# Display sample rows to verify final format and cleaned text with associated labels
data_set.show(n=2, truncate=0)

Traceback (most recent call last):                                  (0 + 1) / 1]
  File "/opt/homebrew/Cellar/apache-spark/4.0.0/libexec/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
    code = worker(sock, authenticated)
  File "/opt/homebrew/Cellar/apache-spark/4.0.0/libexec/python/lib/pyspark.zip/pyspark/daemon.py", line 81, in worker
    worker_main(infile, outfile)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/apache-spark/4.0.0/libexec/python/lib/pyspark.zip/pyspark/worker.py", line 2068, in main
    if read_int(infile) == SpecialLengths.END_OF_STREAM:
       ~~~~~~~~^^^^^^^^
  File "/opt/homebrew/Cellar/apache-spark/4.0.0/libexec/python/lib/pyspark.zip/pyspark/serializers.py", line 597, in read_int
    raise EOFError
EOFError


+----------+---------------------------------------------------------------------------------------------------+-----------------+
|comment_id|lemm_text                                                                                          |hate_speech_score|
+----------+---------------------------------------------------------------------------------------------------+-----------------+
|13128     |thank transmasc present masc look bit gay wonder sorry heckle good hear thing tysm favorite lesbian|-3.92            |
|6626      |flair love woman minority                                                                          |-4.48            |
+----------+---------------------------------------------------------------------------------------------------+-----------------+
only showing top 2 rows


In [33]:
# Use withColumnRenamed to rename 'lemm_text' back to 'text' for compatibility with ML pipelines
renamed_df = data_set.withColumnRenamed("lemm_text", "text")

In [34]:
# Display the schema to verify the renaming of the column
renamed_df.printSchema()

root
 |-- comment_id: integer (nullable = true)
 |-- text: string (nullable = true)
 |-- hate_speech_score: double (nullable = true)



In [35]:
# Display sample rows to confirm renamed column and final content format
renamed_df.show(n=2, truncate=0)

+----------+---------------------------------------------------------------------------------------------------+-----------------+
|comment_id|text                                                                                               |hate_speech_score|
+----------+---------------------------------------------------------------------------------------------------+-----------------+
|13128     |thank transmasc present masc look bit gay wonder sorry heckle good hear thing tysm favorite lesbian|-3.92            |
|6626      |flair love woman minority                                                                          |-4.48            |
+----------+---------------------------------------------------------------------------------------------------+-----------------+
only showing top 2 rows


In [36]:
# Store final preprocessed dataset under consistent variable name
preprocessed_df = renamed_df

## 7) Loading Bluesky API Data

### Set-up

The first block of imports brings in a core set of low-level libraries that act as a "toolkit" for managing the notebook's environment. This phase is not about loading the main application libraries (like atproto or pyspark), but rather about loading the tools necessary to inspect, modify, and prepare the environment for them. Each library serves a specific, foundational purpose:
- sys: Provides access to the Python interpreter. It is used here to get the exact path to the current Python executable (sys.executable), ensuring that any pip commands are run against the correct Python installation tied to this notebook kernel.
- subprocess: It allows the Python script to run external programs, which is needed for executing pip install and pip uninstall commands directly from the code.
- importlib: A module for interacting with Python's import system. Its metadata submodule is used to programmatically check if a package is installed and what its version is.
- gc: The garbage collector is loaded in preparation for the next step, where it will be used to force a cleanup of any library modules that are removed from memory.

In [ ]:
import sys
import subprocess
import importlib.metadata
import importlib
import gc

try:
    importlib.metadata.version("pyspark")
except importlib.metadata.PackageNotFoundError:
    !{sys.executable} -m pip install --quiet pyspark

This section ensures the notebook runs reliably by actively managing critical libraries and their dependencies. The goal is to prevent version conflicts that are common in shared environments like jupyter notebook.

The Managed Libraries and Their Roles:
- atproto: The main library for communicating with the Bluesky network.
- pydantic: A key dependency of atproto, used for validating and structuring API data.
- typing_extensions: A helper library required by pydantic for advanced type hints.
- packaging: A utility that helps manage package versions correctly during the installation process.

So we use packaging to use this installation command. Atproto is the main libarary we need and it depends on pydantic which then depends on typing_extensions.

The Three-Step Process:
1. Uninstall: All listed packages are first removed from the disk (with pip uninstall).
2. Purge from Memory: The packages are then explicitly removed from the active Python session's memory (sys.modules), a critical step to prevent using old, cached versions.
3. Re-install: Finally, atproto and its dependencies are re-installed using specific, known-good versions.


In [ ]:
BaseException
PACKAGES_TO_MANAGE = ["atproto", "pydantic", "typing_extensions", "packaging"]
REQUIRED_VERSIONS = {
    "pydantic": "2.4.0",
    "typing_extensions": "4.12.0",
    "packaging": "20.9"
}

# Uninstalling packages from the disk
uninstall_command = [sys.executable, "-m", "pip", "uninstall", "-y"] + PACKAGES_TO_MANAGE
subprocess.run(uninstall_command, capture_output=True, text=True)

# Deleting packages from memory
modules_to_purge = [m for m in sys.modules if any(p for p in PACKAGES_TO_MANAGE if m == p or m.startswith(p + '.'))]
for module_name in modules_to_purge:
    del sys.modules[module_name]
gc.collect()

# Installing the correct versions of the packages
install_string = f"atproto \"pydantic>={REQUIRED_VERSIONS['pydantic']}\" \"typing_extensions>={REQUIRED_VERSIONS['typing_extensions']}\" \"packaging>={REQUIRED_VERSIONS['packaging']}\""
! "{sys.executable}" -m pip install --upgrade --quiet {install_string}

With the environment now prepared and validated, this final setup phase imports all the necessary components for the main analysis. This block is executed only after the environment has been sanitized, which is the key to its success.

- from atproto import Client, models: This is the payoff for the previous step. By importing atproto after the re-installation, the script makes sure it is loading the newly installed, correct version into memory, not a potentially pre-existing or conflicting one.
- os: The os module is used specifically to read the TIMEFRAME_START_ISO, TIMEFRAME_END_ISO, and FEED_URI_TO_QUERY variables from the system environment, allowing these settings to be configured externally.
- datetime: The datetime module is used to create the default 30-day time window, parse post creation dates from the API, and filter posts to ensure they fall between TIMEFRAME_START_DT and TIMEFRAME_END_DT.
- json: This is needed so that we can access our credentials, which are stored in a seperate JSON file, which can later be included in a .gitignore so that this script can be shared easily

This final set of imports marks the transition from environment setup to application logic. The script is now fully configured with the correct, stable, and verified libraries, ready to connect to the Bluesky API and process data.

In [39]:
import os
from datetime import datetime, timezone, timedelta
import json


# Import the conflicting library (which is now safe)
from atproto import Client, models

With our libraries imported now, the next thing we do is get the behavior of our script organized. This section of code defines our bluesky handle and password and configures all the parameters that we require that will determine what data we fetch from Bluesky, how much data we fetch, and from what time period.

In [40]:
try:
    with open('credentials.json') as config_file:
        config = json.load(config_file)
    
    BLUESKY_HANDLE = config.get("BLUESKY_HANDLE")
    BLUESKY_APP_PASSWORD = config.get("BLUESKY_APP_PASSWORD")
except FileNotFoundError:
    print("ERROR: The 'credentials.json' file was not found in this directory.")

# Next we set parameters for the ammount of posts and how theyre beung fetched
MAX_PAGES_TO_FETCH = 50
POSTS_PER_PAGE = 50 
TARGET_POST_COUNT = 50 

# And lastly we define our time based variables.
default_end_time = datetime.now(timezone.utc) 
default_start_time = default_end_time - timedelta(days=30) 
TIMEFRAME_START_ISO = os.environ.get("TIMEFRAME_START_ISO", default_start_time.isoformat().replace('+00:00', 'Z'))
TIMEFRAME_END_ISO = os.environ.get("TIMEFRAME_END_ISO", default_end_time.isoformat().replace('+00:00', 'Z'))


The last variable we create, is to define where we will stream the data from. Here ewe will make use of blueskys feeds (https://bsky.app/feeds) and decide to use one, that contains popular and new posts

In [41]:
FEED_URI_TO_QUERY = os.environ.get("FEED_URI_TO_QUERY", "at://did:plc:z72i7hdynmk6r22z27h6tvur/app.bsky.feed.generator/hot-classic")

### Helper function to parse ISO time strings

While the previous step defined our time window using ISO-formatted strings (e.g., 2023-10-27T10:00:00Z), these are not directly usable for time-based comparisons in Python. To perform calculations and check if a post falls within our desired range, we need to convert these strings into proper datetime objects.
This code block introduces a robust helper function to perform this conversion and then validates the results to ensure our time window is logical before proceeding.

In [42]:
def parse_iso_datetime(iso_str: str):
    if not iso_str:
        return None
    try:
        if iso_str.endswith('Z'):
            iso_str = iso_str[:-1] + '+00:00'
        dt = datetime.fromisoformat(iso_str)
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt
    except ValueError as e:
        print(f"Warning: Could not parse date '{iso_str}': {e}")
        return None

TIMEFRAME_START_DT = parse_iso_datetime(TIMEFRAME_START_ISO)
TIMEFRAME_END_DT = parse_iso_datetime(TIMEFRAME_END_ISO)

# Error Handling to ensure the date is set correctly, and the start date is before the end date
if not TIMEFRAME_START_DT or not TIMEFRAME_END_DT:
    raise ValueError("Invalid start or end datetime. Please check TIMEFRAME_START_ISO and TIMEFRAME_END_ISO.")
if TIMEFRAME_START_DT >= TIMEFRAME_END_DT:
    raise ValueError("TIMEFRAME_START_DT must be before TIMEFRAME_END_DT.")    

### Accessing Bluesky

With all our configurations set and validated, we are now ready to establish a live connection to the Bluesky network. This code block takes our credentials and uses them to authenticate with the API. A successful login is the final prerequisite before we can begin fetching post data.

The first step is to create an instance of the Client class that we imported from the atproto library. Think of this client object as our personal remote control for the Bluesky API. At this moment, it's a generic, unauthenticated client, ready to be configured with our credentials.

The atproto library sends these credentials to the Bluesky server. If they are correct, the server responds with an authentication token that is stored internally within the client object. From this point forward, any request made using this specific client instance will be automatically authenticated.
If the login is successful, the method also returns a profile object containing public information about our account, which we store for immediate use.

In [43]:
client = Client()
profile = client.login(BLUESKY_HANDLE, BLUESKY_APP_PASSWORD)
print(f"Successfully logged in as {profile.handle} (DID: {profile.did})")
print(f"Client object after login attempt: {client}")

Successfully logged in as user_004.example (DID: did:plc:user001)
Client object after login attempt: <atproto_client.client.client.Client object at 0x31a922a50>


Now that we are logged in, we need a reliable way to process the data we will receive from the Bluesky API. The data for a single post arrives as a complex, nested object. To make this data useful for analysis, we need to extract the key pieces of information and organize them into a simple, flat structure.
This is the purpose of the parse_post_view_data function. It acts as a dedicated "translator," taking a raw PostView object from the API and converting it into a clean, easy-to-use Python dictionary. This approach centralizes our parsing logic, making the main data-fetching loop much cleaner and more readable.

In [44]:
def parse_post_view_data(post_view: models.AppBskyFeedDefs.PostView):
    content = "" #first we define some variables
    created_at_str = ""
    author_did = post_view.author.did
    author_handle = post_view.author.handle

    try: # then we access the individual information snippets we're looking for
        if isinstance(post_view.record, models.AppBskyFeedPost.Record):
            content = post_view.record.text
            created_at_str = post_view.record.created_at
        elif isinstance(post_view.record, dict): 
            content = post_view.record.get('text', '')
            created_at_str = post_view.record.get('createdAt', '')
        else:
            print(f"Warning: Encountered unexpected post record type for URI {post_view.uri}: {type(post_view.record)}")
            return None 

        parsed_date = parse_iso_datetime(created_at_str)
        if parsed_date is None:
             print(f"Warning: Skipping post {post_view.uri} due to unparseable date: '{created_at_str}'")
             return None # Skip if date cannot be parsed

        return {
            "uri": post_view.uri,
            "cid": post_view.cid,
            "author_did": author_did,
            "author_handle": author_handle,
            "content": content,
            "created_at_iso": created_at_str, # Store original string
            "created_at_dt": parsed_date,   # Store parsed datetime object
            "reply_count": post_view.reply_count if post_view.reply_count is not None else 0,
            "repost_count": post_view.repost_count if post_view.repost_count is not None else 0,
            "like_count": post_view.like_count if post_view.like_count is not None else 0,
        }
    except Exception as e:
        print(f"Error parsing post {post_view.uri if hasattr(post_view, 'uri') else 'N/A'}: {e}")
        return None # Return None if any error occurs during parsing

This function acts as the primary engine for our data collection process, bringing together all the components we have previously defined. Its core purpose is to methodically fetch posts from a specific Bluesky feed, such as the main "Following" timeline, by making repeated calls to the API. It operates within a clear set of boundaries: it will stop once it either collects a target number of posts or reaches a pre-defined maximum number of pages to search, whichever comes first. This ensures the script is both efficient and responsible, preventing it from running indefinitely or making an excessive number of requests.
As the function retrieves each page of posts, it acts as a responsible gatekeeper for our data. For every post found, it immediately uses our parse_post_view_data helper function to translate the raw API response into a clean, structured format. More importantly, it checks the timestamp of each post against our specified start and end times, discarding any that fall outside the desired window. The entire process is wrapped in robust error handling, allowing it to gracefully skip over network issues or malformed data without crashing. The final result is a curated list containing only the posts that precisely match our criteria, perfectly prepared for the next stage of analysis.

In [45]:
def get_posts_from_feed_in_timeframe( #we set up the format of some variables 
    client: Client,
    feed_uri: str,
    start_dt: datetime,
    end_dt: datetime,
    target_count: int,
    posts_per_page: int = 50, 
    max_pages: int = 50):
    
    collected_posts = []
    cursor = None
    pages_fetched = 0

    #for ease of use we print the feed uri and timeframe
    print(f"Fetching posts from feed: {feed_uri}") 
    print(f"Looking for posts between {start_dt.isoformat()} and {end_dt.isoformat()}")

    # now we collect posts, as long as we have less posts than our target post count and 
    # the page we're fetching from is less than the highest page we defined
    while len(collected_posts) < target_count and pages_fetched < max_pages: 
        try: #now we make the api call itself
            print(f"Fetching page {pages_fetched + 1} (cursor: {cursor})...")
            params = models.AppBskyFeedGetFeed.Params(
                feed=feed_uri,
                limit=posts_per_page,
                cursor=cursor
            )
            response = client.app.bsky.feed.get_feed(params)

            # now we inculde some error handling to make this function more robust
        except RequestException as e:
            print(f"API Error fetching feed page {pages_fetched + 1}: {e}")
            break 
        except Exception as e:
            print(f"Unexpected error fetching feed page {pages_fetched + 1}: {e}")
            break 

        if not response or not response.feed:
            print("No more posts in the feed or empty response for this page.")
            if pages_fetched == 0:
                print("Consider widening the timeframe or checking the feed URI.")
            break

            # and now we put our post-parsing helper function to use 
            # by passing each individual post through it before fetching the next post
        posts_on_page_in_timeframe = 0
        for feed_view_post in response.feed: 
            if not isinstance(feed_view_post, models.AppBskyFeedDefs.FeedViewPost) or not hasattr(feed_view_post, 'post'):
                 continue 

            post_data = parse_post_view_data(feed_view_post.post)

            if post_data: #and we only look for posts in a certain timeframe
                post_date = post_data["created_at_dt"]
                if start_dt <= post_date <= end_dt:
                    collected_posts.append(post_data)
                    posts_on_page_in_timeframe += 1
                    if len(collected_posts) >= target_count:
                        break 
        print(f"  Found {posts_on_page_in_timeframe} posts in timeframe on this page.")
        print(f"  Total collected so far: {len(collected_posts)} / {target_count}")

        if len(collected_posts) >= target_count: # if we collected enough posts we stop
            print("Target number of posts collected.")
            break

        cursor = response.cursor # if were done with one page, we move to the next and advance the counter
        if not cursor:
            print("No more pages (cursor is null).")
            break
        pages_fetched += 1

    if pages_fetched >= max_pages: #if we have fetched more pages than defined be stop
        print(f"Reached maximum page limit ({max_pages}) before collecting {target_count} posts.")
    elif len(collected_posts) < target_count:
        print(f"Finished fetching all available pages within the limit ({max_pages}), but only collected {len(collected_posts)}/{target_count} posts in the specified timeframe.")

    return collected_posts

This final block executes the data pipeline. After first verifying the Bluesky client is authenticated, it calls our main function to fetch posts according to the specified timeframe and feed criteria. The resulting list of posts is then immediately converted into a Spark Resilient Distributed Dataset (RDD), preparing it for large-scale, parallel analysis. The script concludes by printing a summary and a sample of the RDD's contents, providing immediate confirmation that the data has been successfully ingested into the Spark ecosystem.

In [46]:
if 'client' in locals() and client is not None and client.me:
    try: # first we set the variables 
        sampled_posts = get_posts_from_feed_in_timeframe(
            client,
            feed_uri=FEED_URI_TO_QUERY,
            start_dt=TIMEFRAME_START_DT,
            end_dt=TIMEFRAME_END_DT,
            target_count=TARGET_POST_COUNT,
            posts_per_page=POSTS_PER_PAGE,
            max_pages=MAX_PAGES_TO_FETCH
        )
        print(f"\n--- Collected {len(sampled_posts)} Posts ---") #the we use print as a helper
        if sampled_posts: #now we convert the posts to an rdd
            if 'spark' in globals() and sampled_posts:
                 posts_rdd = spark.sparkContext.parallelize(sampled_posts) 
                 print("\n--- RDD created from collected posts ---")
                 print(f"Number of partitions in RDD: {posts_rdd.getNumPartitions()}")

                 print("\nSample RDD elements:")
                 for i, post in enumerate(posts_rdd.take(5)): # we take the first 5 elements and show them
                     print(f"  {i+1}. {post}")

                    # and lastly we include some error handling with easy to understand errors
            elif sampled_posts:
                 print("SparkSession not initialized. Skipping RDD creation.")
            else:
                print("No posts collected, RDD not created.")

        else:
            print("No posts collected that fit the criteria.")

    except Exception as e:
        print(f"An error occurred during the post fetching process: {e}")

else:
    print("Client not authenticated. Cant fetch posts.")

Fetching posts from feed: at://did:plc:z72i7hdynmk6r22z27h6tvur/app.bsky.feed.generator/hot-classic
Looking for posts between 2025-06-06T18:09:32.717531+00:00 and 2025-07-06T18:09:32.717531+00:00
Fetching page 1 (cursor: None)...
  Found 50 posts in timeframe on this page.
  Total collected so far: 50 / 50
Target number of posts collected.

--- Collected 50 Posts ---

--- RDD created from collected posts ---
Number of partitions in RDD: 1

Sample RDD elements:
  1. {'uri': 'at://did:plc:user002/app.bsky.feed.post/3ltcuhymqas2g', 'cid': 'bafyreihzbh67yjzmovygmjryv4m3du7oa3xylh55q3zxvhxzz76ginserq', 'author_did': 'did:plc:user002', 'author_handle': 'user_002.example', 'content': 'Ken Burns calls it "shortsighted" to eliminate funding for the Corporation for Public Broadcasting\n\nwww.cbsnews.com/news/ken-bur...', 'created_at_iso': '2025-07-06T18:08:06.520Z', 'created_at_dt': datetime.datetime(2025, 7, 6, 18, 8, 6, 520000, tzinfo=datetime.timezone.utc), 'reply_count': 1, 'repost_count': 9

## 8) Preprocessing Bluesky API Data
In this step, we prepare the posts collected from the Bluesky API to prepare them for further analysis. We convert the raw dictionary into a Spark DataFrame, select only the columns relevant to our task, namely the uri  and content features, which represent the unique post ID and the actual text and remove all entries without usable content. We only select these two features because they are essential for our NLP and prediction pipeline. All other metadata is excluded to simplify processing and to avoid collecting unnecessary personal or platform-specific information.

**Libraries used:**
- `pyspark.sql.Row` – for converting RDDs into Spark DataFrames
- `pyspark.sql.functions` – for filtering and cleaning the content

In [47]:
from pyspark.sql import Row

# Try to convert each dictionary in the RDD into a Row to prevent schema inference warnings
# This step ensures that the structure is consistent for DataFrame creation
try:
    posts_rdd_row = posts_rdd.map(lambda x: Row(**x))
    post_df = spark.createDataFrame(posts_rdd_row)
except Exception as e:
    print("Error while converting RDD to DataFrame:", e)

In [48]:
# Display the structure of the DataFrame including column names and data types
post_df.printSchema()

# Display the first 2 rows of the DataFrame to verify correct conversion from RDD
post_df.show(2)

root
 |-- uri: string (nullable = true)
 |-- cid: string (nullable = true)
 |-- author_did: string (nullable = true)
 |-- author_handle: string (nullable = true)
 |-- content: string (nullable = true)
 |-- created_at_iso: string (nullable = true)
 |-- created_at_dt: timestamp (nullable = true)
 |-- reply_count: long (nullable = true)
 |-- repost_count: long (nullable = true)
 |-- like_count: long (nullable = true)

+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+------------+----------+
|                 uri|                 cid|          author_did|       author_handle|             content|      created_at_iso|       created_at_dt|reply_count|repost_count|like_count|
+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+------------+----------+
|at://did:plc:user002...|b

In [49]:
# Select only the relevant columns 'uri' and 'content' for our analysis
# We include a try-block in case one of the columns does not exist (e.g. due to schema changes or earlier processing errors)
try:
    bsky_df_selected = post_df.select("uri", "content")

    # Display the schema to confirm that only the selected columns are retained
    bsky_df_selected.printSchema()

    # Show first 50 rows (we loaded exactly 50 posts from the API) to verify usable content
    bsky_df_selected.show(50)

except Exception as e:
    print("Failed to select or display the columns 'uri' and 'content':", e)


root
 |-- uri: string (nullable = true)
 |-- content: string (nullable = true)

+--------------------+--------------------+
|                 uri|             content|
+--------------------+--------------------+
|at://did:plc:user002...|Ken Burns calls i...|
|at://did:plc:user003...|If he was too fre...|
|at://did:plc:user004...|               Um...|
|at://did:plc:user005...|what people think...|
|at://did:plc:user006...|Emergency alerts ...|
|at://did:plc:user009...|“The use of GLP-1...|
|at://did:plc:user010...|I mean he’s my co...|
|at://did:plc:user011...|When the music fa...|
|at://did:plc:user012...|A luta contra a t...|
|at://did:plc:user013...|Last month, White...|
|at://did:plc:user014...|*stares in Paul V...|
|at://did:plc:user015...|POV my plump nuts...|
|at://did:plc:user016...|Legs are shaky fr...|
|at://did:plc:user017...|Some people aren'...|
|at://did:plc:user018...|You basically can...|
|at://did:plc:user019...|Remember that the...|
|at://did:plc:user020...|  help he's

In [50]:
from pyspark.sql import functions as F

# Remove rows where content is null or contains only whitespace
bsky_df = bsky_df_selected.filter(bsky_df_selected.content.isNotNull() & (F.length(F.trim(bsky_df_selected.content)) > 0))

# Count rows before and after empty content
original_count_bsky = bsky_df_selected.count()
cleaned_count_bsky = bsky_df.count()

print(f"Rows before removing empty content: {original_count_bsky}")
print(f"Rows after removing empty content: {cleaned_count_bsky}")

Rows before removing empty content: 50
Rows after removing empty content: 47


## 9) NLP for Bluesky API Data
In this step, we apply the same workflow to the Bluesky API data as we did for the Huggingface dataset. This includes language filtering, removing stopwords and special characters, part-of-speech tagging, lemmatization, and handling empty or duplicate entries. At the end, we keep only the cleaned content and the uri

The structure and logic of this step were provided in the course *Data Processing 2: Scalable Data Processing, Legal & Ethical Foundations of Data Science* (WU Vienna) by Ines Akaichi and were adapted to fit the needs of our project setup.

In [51]:
# Use "check_lang_udf" to predict the language of each comment
# The result is stored in a new column called "lang" using withColumn
# This allows us to keep the original text while adding the language prediction for further filtering
bsky_lang_df = bsky_df.withColumn("lang", check_lang_udf(bsky_df["content"]))

# Filter the dataset to keep only entries classified as English
# Only rows with at least 90 percent probability for English are retained
bsky_en_df = bsky_lang_df.filter(bsky_lang_df["lang"] == "en")

In [52]:
# Display 10 rows from the Bluesky API data with predicted language
bsky_lang_df.show(n=10)

# Count how many comments were classified as English
print("Number of englisch comments:", bsky_en_df.count())

# Show 2 example rows from the filtered English-only comments
bsky_en_df.show(2)

+--------------------+--------------------+----+
|                 uri|             content|lang|
+--------------------+--------------------+----+
|at://did:plc:user002...|Ken Burns calls i...|  en|
|at://did:plc:user003...|If he was too fre...|  en|
|at://did:plc:user004...|               Um...|  NA|
|at://did:plc:user005...|what people think...|  en|
|at://did:plc:user006...|Emergency alerts ...|  en|
|at://did:plc:user009...|“The use of GLP-1...|  en|
|at://did:plc:user010...|I mean he’s my co...|  en|
|at://did:plc:user011...|When the music fa...|  en|
|at://did:plc:user012...|A luta contra a t...|  pt|
|at://did:plc:user013...|Last month, White...|  en|
+--------------------+--------------------+----+
only showing top 10 rows


Number of englisch comments: 34


+--------------------+--------------------+----+
|                 uri|             content|lang|
+--------------------+--------------------+----+
|at://did:plc:user002...|Ken Burns calls i...|  en|
|at://did:plc:user003...|If he was too fre...|  en|
+--------------------+--------------------+----+
only showing top 2 rows


In [53]:
# Use 'remove_stops_udf' to create a new column without stopwords
# This reduces dimensionality by removing common words that carry little meaning
# We use withColumn to add the new cleaned content column to the DataFrame
bsky_rm_stops_df = bsky_en_df.withColumn("stop_text", remove_stops_udf(bsky_en_df["content"]))

In [54]:
# Display original and cleaned content to verify stopword removal
bsky_rm_stops_df.select("content","stop_text").show(n=2, truncate=0)

+-------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------+
|content                                                                                                                              |stop_text                                                                                                       |
+-------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------+
|Ken Burns calls it "shortsighted" to eliminate funding for the Corporation for Public Broadcasting\n\nwww.cbsnews.com/news/ken-bur...|Ken Burns calls "shortsighted" eliminate funding Corporation Public Broadcasting www.cbsnews.com/news/ken-bur...|
|If 

In [55]:
# Use 'remove_features_udf' to clean up additional unwanted tokens 
# This includes for example custom patterns like URLs or user mentions to reduce noise
# We use withColumn to add the new cleaned text column to the DataFrame
bsky_rm_features_df = bsky_rm_stops_df.withColumn("feat_text", remove_features_udf(bsky_rm_stops_df["stop_text"]))

In [56]:
# Display text before and after feature removal to verify cleaning step
bsky_rm_features_df.select("stop_text","feat_text").show(n=2, truncate=0)

+----------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------+
|stop_text                                                                                                       |feat_text                                                                                                  |
+----------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------+
|Ken Burns calls "shortsighted" eliminate funding Corporation Public Broadcasting www.cbsnews.com/news/ken-bur...|ken burns calls shortsighted eliminate funding corporation public broadcasting www cbsnews com news ken bur|
|If freaking ______ see everyone else world saw, freaking ______ listened now.                              

In [57]:
# Use 'tag_and_remove_udf' to tag remaining words and keep only nouns, verbs and adjectives
# We use withColumn to add the POS-tagged and filtered text as a new column
bsky_tagged_df = bsky_rm_features_df.withColumn("tagged_text", tag_and_remove_udf(bsky_rm_features_df["feat_text"]))

In [58]:
# Display text before and after POS tagging to verify filtering of unwanted word types
bsky_tagged_df.select("feat_text","tagged_text").show(n=2, truncate=0)

+-----------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------+
|feat_text                                                                                                  |tagged_text                                                                                                  |
+-----------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------+
|ken burns calls shortsighted eliminate funding corporation public broadcasting www cbsnews com news ken bur| ken burns calls shortsighted eliminate funding corporation public broadcasting www cbsnews com news ken bur |
|  freaking see everyone else world saw freaking listened now                                               | freaking s

In [59]:
# Use 'lemmatize_udf' to reduce words to their base forms for better consistency
# This helps reduce dimensionality and improves feature alignment across similar terms
# We use withColumn to add the lemmatized version of the tagged text to the DataFrame
bsky_lemm_df = bsky_tagged_df.withColumn("lemm_text", lemmatize_udf(bsky_tagged_df["tagged_text"]))

In [60]:
# Display text before and after lemmatization to verify word normalization
bsky_lemm_df.select("tagged_text","lemm_text").show(n=2, truncate=0)

+-------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------+
|tagged_text                                                                                                  |lemm_text                                                                                                |
+-------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------+
| ken burns calls shortsighted eliminate funding corporation public broadcasting www cbsnews com news ken bur |ken burn call shortsighted eliminate funding corporation public broadcasting www cbsnews com news ken bur|
| freaking see everyone else world saw freaking listened                                                      |freak see everyon

In [61]:
# Use 'check_blanks_udf' to detect rows with only blank spaces in the lemmatized text
# This step helps ensure that no empty or meaningless inputs are passed into later ML processing
# We use withColumn to add a helper column and then filter out the blank rows accordingly
bsky_check_blanks_df = bsky_lemm_df.withColumn("is_blank", check_blanks_udf(bsky_lemm_df["lemm_text"]))
bksy_no_blanks_df = bsky_check_blanks_df.filter(bsky_check_blanks_df["is_blank"] == "False")

In [62]:
# Display 'is_blank' values to verify all empty rows were removed
bksy_no_blanks_df.select("is_blank").show(n=2, truncate=0)

+--------+
|is_blank|
+--------+
|False   |
|False   |
+--------+
only showing top 2 rows


In [63]:
# Use 'dropDuplicates' to remove duplicate rows from the DataFrame  
# This is based on identical 'text' and 'hate_speech_score' values  
# Ensures data quality by avoiding multiple entries of the same sample
bsky_dedup_df = bksy_no_blanks_df.dropDuplicates(['content', 'uri'])

In [64]:
# Display final schema to verify structure and included columns after cleaning steps
bsky_dedup_df.printSchema()

root
 |-- uri: string (nullable = true)
 |-- content: string (nullable = true)
 |-- lang: string (nullable = true)
 |-- stop_text: string (nullable = true)
 |-- feat_text: string (nullable = true)
 |-- tagged_text: string (nullable = true)
 |-- lemm_text: string (nullable = true)
 |-- is_blank: string (nullable = true)



In [65]:
# Use 'select' to create a final dataset with only the relevant columns 
# This includes comment_id as identifier, lemmatized text for analysis, and the hate speech score as target
bsky_data_set = bsky_dedup_df.select(bsky_dedup_df['uri'], bsky_dedup_df['lemm_text'])

In [66]:
# Display the schema of the final dataset to verify column names and data types
bsky_data_set.printSchema()

root
 |-- uri: string (nullable = true)
 |-- lemm_text: string (nullable = true)



In [67]:
# Display sample rows to verify final format and cleaned text with associated labels
bsky_data_set.show(n=2, truncate=0)

+----------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------+
|uri                                                                   |lemm_text                                                                                                                               |
+----------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------+
|at://did:plc:user007/app.bsky.feed.post/3ltcu6ius622m|authority need plan retain rainwater fall say hetherington upland area sink ground river replenish reservoir water table www com content|
|at://did:plc:user008/app.bsky.feed.post/3ltcubjh5rc2j|kickflip happy pubby get new badge amaze bsky social thank                                                                

In [68]:
# Use withColumnRenamed to rename 'lemm_text' back to 'content' for compatibility with ML pipelines
bsky_renamed_df = bsky_data_set.withColumnRenamed("lemm_text", "content")

In [69]:
# Display the schema to verify the renaming of the column
bsky_renamed_df.printSchema()

root
 |-- uri: string (nullable = true)
 |-- content: string (nullable = true)



In [70]:
# Display sample rows to confirm renamed column and final content format
bsky_renamed_df.show(n=2, truncate=0)

+----------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------+
|uri                                                                   |content                                                                                                                                 |
+----------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------+
|at://did:plc:user007/app.bsky.feed.post/3ltcu6ius622m|authority need plan retain rainwater fall say hetherington upland area sink ground river replenish reservoir water table www com content|
|at://did:plc:user008/app.bsky.feed.post/3ltcubjh5rc2j|kickflip happy pubby get new badge amaze bsky social thank                                                                

In [71]:
# Store final preprocessed dataset under consistent variable name
bsky_preprocessed_df = bsky_renamed_df

## 10) Preprocessing for Machine Learning

Now that the data has been cleaned and has gone through natural language processing, we need to do some further processing to prepare it for machine learning. One major change that needs to be made, is changing the text to something that machines can parse. This is done using the methods discussed in class with Ines Akaichi - namely RegexTokenizer, HashingTF and IDF.  The former splits the text into tokens based on expression patterns, then HashingTF converts these into numeric token counts which are then scaled by the latter program. We add all of these to a pipeline and apply it to the training and test data.  

In [72]:
#import packages
from pyspark.ml.feature import Tokenizer, HashingTF, IDF
from pyspark.ml import Pipeline

#tokenizing text such that data is suitable for machine learning
tokenizer = Tokenizer(inputCol="text", outputCol="words")
hashingTF = HashingTF(inputCol="words", outputCol="rawFeatures", numFeatures=1000)
idf = IDF(inputCol="rawFeatures", outputCol="features")
pipeline = Pipeline(stages=[tokenizer, hashingTF, idf])

#split dataset into train and test
train_data, test_data = preprocessed_df.randomSplit([0.8, 0.2], seed=666)

#fit to pipeline
train_df = pipeline.fit(train_data) 
transformed_train = train_df.transform(train_data)
transformed_train.show(3)

#same for test data
test_df = pipeline.fit(test_data)
transformed_test = test_df.transform(test_data)
transformed_test.show(3)

+----------+--------------------+-----------------+--------------------+--------------------+--------------------+
|comment_id|                text|hate_speech_score|               words|         rawFeatures|            features|
+----------+--------------------+-----------------+--------------------+--------------------+--------------------+
|         1|  white people trash|             0.46|[white, people, t...|(1000,[263,312,84...|(1000,[263,312,84...|
|         2|caught big gay kn...|             0.03|[caught, big, gay...|(1000,[48,100,155...|(1000,[48,100,155...|
|         3|say say say find ...|            -1.29|[say, say, say, f...|(1000,[24,125,146...|(1000,[24,125,146...|
+----------+--------------------+-----------------+--------------------+--------------------+--------------------+
only showing top 3 rows


+----------+--------------------+-----------------+--------------------+--------------------+--------------------+
|comment_id|                text|hate_speech_score|               words|         rawFeatures|            features|
+----------+--------------------+-----------------+--------------------+--------------------+--------------------+
|        35|joe rogan listen ...|             2.37|[joe, rogan, list...|(1000,[14,48,50,6...|(1000,[14,48,50,6...|
|        48|give tired poor r...|             1.87|[give, tired, poo...|(1000,[103,142,15...|(1000,[103,142,15...|
|        57|anon betamale cut...|              0.7|[anon, betamale, ...|(1000,[9,206,654,...|(1000,[9,206,654,...|
+----------+--------------------+-----------------+--------------------+--------------------+--------------------+
only showing top 3 rows


## 11) Machine learning

For the modeling, we decided to use ones that can handle complex underlying relationships, and one model as a base-line.
**Models used:**
- `Random Forest Regressor` – a model that uses an ensemble of decision trees and takes the average result
- `Gradient Boosted Trees` – a model that also uses trees, but minimizes a loss function recursively
- `Linear Regression` – a simple linear model as a base line

Once the models are fit to the training dataset, we use them to predict the hate speech score of the test data.

In [73]:
#import pakages
from pyspark.ml.regression import RandomForestRegressor, GBTRegressor

#create first model using random forest regressor
regressor = RandomForestRegressor(labelCol="hate_speech_score", featuresCol="features", numTrees=10)
model_1 = regressor.fit(transformed_train)

#create predictions
predictions_1 = model_1.transform(transformed_test)
predictions_1.select("text", "prediction").show(3)

+--------------------+-------------------+
|                text|         prediction|
+--------------------+-------------------+
|joe rogan listen ...|-0.3670205468963681|
|give tired poor r...|-1.1497414797151184|
|anon betamale cut...|-1.1497414797151184|
+--------------------+-------------------+
only showing top 3 rows


In [74]:
#creating second model using gradient boosted tree regression
regressor2 = GBTRegressor(labelCol="hate_speech_score", featuresCol="features", maxIter=10)
model_2 = regressor2.fit(transformed_train)

predictions_2 = model_2.transform(transformed_test)
predictions_2.select("text", "prediction").show(3)

+--------------------+-------------------+
|                text|         prediction|
+--------------------+-------------------+
|joe rogan listen ...| 0.5627071753950873|
|give tired poor r...|-1.1352508551235883|
|anon betamale cut...|-0.9138279457096079|
+--------------------+-------------------+
only showing top 3 rows


25/07/06 20:21:52 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


In [75]:
#creating third base-line model: linear regression
from pyspark.ml.regression import LinearRegression
lr = LinearRegression(labelCol="hate_speech_score", featuresCol="features")
model_3 = lr.fit(transformed_train)

predictions_3 = model_3.transform(transformed_test)
predictions_3.select("text", "prediction").show(3)

25/07/06 20:24:26 WARN Instrumentation: [33d5a868] regParam is zero, which might cause numerical instability and overfitting.
25/07/06 20:24:26 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


+--------------------+-------------------+
|                text|         prediction|
+--------------------+-------------------+
|joe rogan listen ...| 1.2053147659577765|
|give tired poor r...|-0.5257006311806383|
|anon betamale cut...|-0.3478690206847457|
+--------------------+-------------------+
only showing top 3 rows


## 12) Model evaluation

In this step, we aim to visualize our results, by first displaying top hate speech posts using the `col` package. Then, we want to evaluate how well these models did. One of the main ways, is by calculating the root mean square error between the predicted and actual hate speech score. To determine goodness of model fit, we also calculate Pearson's R^2.

In [76]:
#show 5 highest hatespeech comments according to models
#model 1
from pyspark.sql.functions import col

top_hate_speech_1 = predictions_1.select("text",  "prediction") \
    .orderBy(col("prediction").desc())

top_hate_speech_1.show(3, truncate=False)

#model 2
top_hate_speech_2 = predictions_2.select("text", "prediction") \
    .orderBy(col("prediction").desc())

top_hate_speech_2.show(3, truncate=False)

#model 3
top_hate_speech_3 = predictions_3.select("text", "prediction") \
    .orderBy(col("prediction").desc())

top_hate_speech_3.show(3, truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------+------------------+
|text                                                                                                                         |prediction        |
+-----------------------------------------------------------------------------------------------------------------------------+------------------+
|bitch graduation star fuck hella fuckin gay faggot as bitch suck gang dick bitch gotta nigga mode slap dusty as ashy as bitch|0.9964444218310641|
|hope autistic filthy offensive susan shit pant cry bitch susan suck amy schumers fat pig titty milk                          |0.9778179309062762|
|get fuck normie faggot cut street road full effect fuck lift bro                                                             |0.9649572371830082|
+---------------------------------------------------------------------------------------------------------------------

+--------------------------------------------------------+------------------+
|text                                                    |prediction        |
+--------------------------------------------------------+------------------+
|fuck cunt do shit kill faggot                           |3.6385682062127107|
|let kill woman dare suck dick fuck disgust whore obvs   |2.5918645266079796|
|bitch kill hoe pride dignity lettin mf fuck wit cucumber|2.1521328498650303|
+--------------------------------------------------------+------------------+
only showing top 3 rows


+-------------------------------------------------------------------------------------------------------------------------------------------------+-----------------+
|text                                                                                                                                             |prediction       |
+-------------------------------------------------------------------------------------------------------------------------------------------------+-----------------+
|fuck little dirty whore bet fucker dick way pussy look tell get fuck chance get fuck nasty slut stick head fuck toilet wife fuck as mikedance kik|5.937307684211095|
|hey faggot faggot jew nigger faggot kike jew nigger sub bruh develop stable secure country evil suppressive huwhite man gay url                  |5.606961168064514|
|bitch graduation star fuck hella fuckin gay faggot as bitch suck gang dick bitch gotta nigga mode slap dusty as ashy as bitch                    |4.768170651371448|
+---

In [77]:
#evaluating root mean square error
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(labelCol="hate_speech_score", predictionCol="prediction", metricName="rmse")

rmse_1 = evaluator.evaluate(predictions_1)
rmse_2 = evaluator.evaluate(predictions_2)
rmse_3 = evaluator.evaluate(predictions_3)

print(f"Random Forest RMSE: {rmse_1}")
print(f"Gradient Boosted Tree RMSE: {rmse_2}")
print(f"Linear Regression RMSE: {rmse_3}")

Random Forest RMSE: 1.8889063839721523
Gradient Boosted Tree RMSE: 1.8238915160333178
Linear Regression RMSE: 1.686502481550834


In [78]:
#get r squared values
print(evaluator.setMetricName("r2").evaluate(predictions_1))
print(evaluator.setMetricName("r2").evaluate(predictions_2))
print(evaluator.setMetricName("r2").evaluate(predictions_3))

0.16354672235545398


0.22013608635455695


0.3332012337593504


**Analysis of Results**

According to both RMSE and R^2, the simple linear model seemed to perform the best. Even so, the RMSE for the linear model is almost 15% of the entire range. A similar issue appears in the R^2 values, as all of them are rather small, meaning the model is not doing much better than just simply predicting the mean. Both of these signify underfitting, which is likely due to a complex underlying relationship in the data. Moreover, it could be that the relationship between hate speech score and text is not that strong, and other factors, such as user id, may play a role. Nonetheless, we shall move forward with the linear model, as it performed the best. Then, we used our best model, the linear one, to predict the hate speech scores of the posts retrieved from Bluesky. To preserve the original posts through the pre-processing, we created row ids so that the original text content of the posts can be printed along with the predicted hate speech scores.  

## 13) Bluesky Predictions

Now, we will predict the Bluesky data using the linear model. To preserve the original posts, we will give the rows ids using `monotonically_increasing_id`

In [79]:
#now preprocess then predict the bluesky data set
from pyspark.sql.functions import monotonically_increasing_id

bsky_preprocessed_df = bsky_preprocessed_df.withColumnRenamed("content", "text")
bsky_preprocessed_df = bsky_preprocessed_df.withColumn("row_id", monotonically_increasing_id())
bsky_df = pipeline.fit(bsky_preprocessed_df)
bluesky_test = bsky_df.transform(bsky_preprocessed_df)
bluesky_test.show(3)

predictions_bluesky = bluesky_test.select("features", "row_id")
predictions_bluesky = model_3.transform(bluesky_test)
predictions_bluesky.select("text", "prediction").show(3)

+--------------------+--------------------+------+--------------------+--------------------+--------------------+
|                 uri|                text|row_id|               words|         rawFeatures|            features|
+--------------------+--------------------+------+--------------------+--------------------+--------------------+
|at://did:plc:user007...|authority need pl...|     0|[authority, need,...|(1000,[12,123,176...|(1000,[12,123,176...|
|at://did:plc:user008...|kickflip happy pu...|     1|[kickflip, happy,...|(1000,[141,347,42...|(1000,[141,347,42...|
|at://did:plc:user049...|imagine sun fill ...|     2|[imagine, sun, fi...|(1000,[41,83,150,...|(1000,[41,83,150,...|
+--------------------+--------------------+------+--------------------+--------------------+--------------------+
only showing top 3 rows
+--------------------+-------------------+
|                text|         prediction|
+--------------------+-------------------+
|authority need pl...|-0.565373012006158

In [80]:
# Add this line right before your failing code to see the available columns
bsky_preprocessed_df.printSchema()

root
 |-- uri: string (nullable = true)
 |-- text: string (nullable = true)
 |-- row_id: long (nullable = false)



We now join the two PySpark DataFrames, explicitly select columns to resolve name ambiguity from the join, sorts the results by prediction score in descending order, and then displays the top 5 predictions. For that to work properly we need to import the col function from pyspark, so that the dataframes can connect properly.

In [81]:
from pyspark.sql.functions import col

# Joining the Dataframes
output = predictions_bluesky.join(bsky_preprocessed_df, on="row_id")

# Printing of the schema for troubleshooting.
print("Schema of the joined 'output' DataFrame:")
output.printSchema()

# Now we select the columns for the final result.
# We explicitly tell Spark which dataframe to take the column from to remove ambiguity in the output dataframe
top_hate_speech_bluesky = output.select(
    predictions_bluesky["text"].alias("content"),  
    predictions_bluesky["prediction"]              
).orderBy(col("prediction").desc())


# Lastly we print the top 5 predictions
print("\nTop 5 predictions:")
top_hate_speech_bluesky.show(5, truncate=False)

Schema of the joined 'output' DataFrame:
root
 |-- row_id: long (nullable = false)
 |-- uri: string (nullable = true)
 |-- text: string (nullable = true)
 |-- words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- rawFeatures: vector (nullable = true)
 |-- features: vector (nullable = true)
 |-- prediction: double (nullable = false)
 |-- uri: string (nullable = true)
 |-- text: string (nullable = true)


Top 5 predictions:
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------+
|content                                                                                                                                                                |prediction           |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------

**Interpretation of Results**

Looking at the final outputs, despite low R^2 values, the predicted values seem to match up to the content of the posts, for example the, the last post about rainwater needing to be retained, has a hate speech score of ~ –0.56, meaning it is neutral speech, which makes sense. Overall, it seems there was not really any hate speech in the Bluesky posts, according to the predictions, since the highest score is ~0, which still falls into the category of neutral or ambiguous. 